# 02 · Trích đặc trưng, huấn luyện head của các chuyên gia, xuất bundle

Thay cho các lệnh `src.models.extract`, `src.models.heads`, `src.models.bundle` trong README.

| Bước | Thời gian (RTX 4060) | Ghi chú |
|---|---|---|
| Trích embedding CLIP ViT-L/14 | ~25 phút | cần GPU + tải trọng số CLIP |
| Trích embedding DINOv2 ViT-B/14 | ~10 phút | |
| Head tuyến tính CLIP/DINOv2 | vài giây | trên embedding đã cache |
| CNN tần số (SRM) | vài phút | huấn luyện trên ảnh |
| Bundle cho từng split × biến thể head | vài phút | |

Notebook này cần dữ liệu thật (notebook 01). **Ở demo mode** các bước này được `build_demo` làm thay,
nên bạn có thể bỏ qua notebook này và chạy thẳng notebook 03/04.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, Image, display

from src.core.config import load_config
from src.core.contracts import SplitManifest

pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 40)
cfg = load_config()
print("repo:", ROOT)

In [ ]:
import torch
MANIFEST = "data/normalized_v2/manifest.json"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HEAD_VARIANTS = ["clean", "aug"]
SPLITS = ["train_experts", "val_gating_fit", "val_gating_verify", "test_benchmark", "s4_support", "s4_eval"]
RUN = Path(MANIFEST).exists()
print("device:", DEVICE, "| manifest found:", RUN)

## 1. Trích embedding (frozen backbone), cache theo sha256, chạy tiếp được nếu bị ngắt

In [ ]:
from src.models.extract import main as extract_main
if RUN:
    for expert in ("clip", "dinov2"):
        extract_main(["--manifest", MANIFEST, "--expert", expert, "--device", DEVICE])

## 2. Huấn luyện head cho từng chuyên gia

`clean` = chỉ ảnh sạch; `aug` = ảnh sạch + biến thể blur/JPEG ngẫu nhiên (train_aug).
Với chuyên gia tần số, `--select-freq` chọn đầu vào (SRM hoặc log-FFT) trên val_gating_fit.

In [ ]:
from src.models.heads import main as heads_main
if RUN:
    for hv in HEAD_VARIANTS:
        for expert in ("clip", "dinov2"):
            heads_main(["--manifest", MANIFEST, "--expert", expert, "--variant", hv, "--device", DEVICE])
        heads_main(["--manifest", MANIFEST, "--expert", "freq", "--variant", hv, "--select-freq", "--device", DEVICE])

## 3. Xuất bundle (logit, cosine distance, đặc trưng chất lượng) cho tầng gating

In [ ]:
from src.models.bundle import main as bundle_main
if RUN:
    for hv in HEAD_VARIANTS:
        for split in SPLITS:
            bundle_main(["--manifest", MANIFEST, "--split", split, "--head-variant", hv, "--device", DEVICE])

## 4. Kiểm tra nhanh: AUROC của từng chuyên gia trên val_gating_fit

In [ ]:
from sklearn.metrics import roc_auc_score
from src.core.contracts import EXPERT_NAMES
from src.learning.bundle_io import bundle_path, load_split
rows = []
for hv in HEAD_VARIANTS:
    if not bundle_path(cfg, "val_gating_fit", hv).exists():
        continue
    b = load_split(cfg, "val_gating_fit", hv)
    clean = b.clean_mask()
    for j, e in enumerate(EXPERT_NAMES):
        rows.append({"head": hv, "expert": e, "auroc_clean": roc_auc_score(b.label[clean], b.logits[clean, j]),
                     "auroc_all": roc_auc_score(b.label, b.logits[:, j])})
display(pd.DataFrame(rows) if rows else Markdown("_Chưa có bundle: chạy các bước ở trên với dữ liệu thật._"))